## Data reduction

In [ ]:
!ls -la /net/vega/data/users/observatory/images/260429/STL-6303E/i/

In [ ]:
import os
import shutil
import glob
# paths
src = '/net/vega/data/users/observatory/images/260429/STL-6303E/i/'
dst = '/net/virgo01/data/users/ahayr/meine_beobachtungen/'

if not os.path.exists(dst):
    os.makedirs(dst)

files = glob.glob(os.path.join(src, '*.FIT'))

# fallback if extensions are lowercase
if not files:
    files = [os.path.join(src, f) for f in os.listdir(src) if f.lower().endswith(('.fit', '.fits'))]

print(f"found {len(files)} files, copying...")

for p in files:
    name = os.path.basename(p)
    
    # skip that weird 'Select Folder' file
    if "Select Folder" in name:
        continue
        
    shutil.copy(p, os.path.join(dst, name))

print("done.")
print(os.listdir(dst)[:5])

In [ ]:
import os
import re
import pandas as pd
from astropy.io import fits

dst = '/net/virgo01/data/users/ahayr/meine_beobachtungen/'
data = []

for f in sorted(os.listdir(dst)):
    if f.lower().endswith(('.fit', '.fits')):
        
        match = re.search(r'\.(\d{8})\.', f)
        if match:
            num = int(match.group(1))
            if num < 145:
                continue
        else:
            continue
            
        full_path = os.path.join(dst, f)
        try:
            with fits.open(full_path) as hdul:
                hdr = hdul[0].header
                if (num > 256) and (num < 267):
                    hdr["IMAGETYP"] = "Dark Frame"
                data.append({
                    'filename': f,
                    'object': hdr.get('OBJECT', 'N/A'),
                    'type': hdr.get('IMAGETYP', 'N/A'),
                    'exptime': hdr.get('EXPTIME', 'N/A'),
                    'filter': hdr.get('FILTER', 'N/A')
                })
        except Exception as e:
            print(f"err on {f}: {e}")

df = pd.DataFrame(data)

# extract file number to drop duplicates
df['tmp_num'] = df['filename'].str.extract(r'\.(\d{8})\.')
df = df.drop_duplicates(subset=['tmp_num']).drop(columns=['tmp_num'])

pd.set_option('display.max_rows', None)

print(f"done. unique rows: {len(df)}")
display(df)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

bias_frames = []

print("loading bias frames...")
for idx, row in df.iterrows():
    # check both header type and filename for 'bias'
    if 'BIAS' in str(row['type']).upper() or 'BIAS' in str(row['filename']).upper():
        path = os.path.join(dst, row['filename'])
        try:
            with fits.open(path) as hdul:
                bias_frames.append(hdul[0].data)
        except Exception as e:
            print(f"err loading {row['filename']}: {e}")

# combine to master if we found any
if bias_frames:
    # median combine to kick out cosmic rays / alpha hits
    master_bias = np.median(bias_frames, axis=0)
    print(f"master bias done using {len(bias_frames)} frames.")
    
    # plot with 2-98 percentile scaling to see structures
    plt.figure(figsize=(8, 6))
    plt.imshow(master_bias, cmap='gray', origin='lower', 
               vmin=np.percentile(master_bias, 2), vmax=np.percentile(master_bias, 98))
    plt.colorbar(label='ADU')
    plt.title('Master Bias')
    plt.show()
else:
    print("no bias frames found. check df filtering.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

dark_frames = []
exposure_time = []
separated_darks = []

print("loading dark frames...")
for idx, row in df.iterrows():
    # check both header type and filename for 'dark'
    if 'DARK' in str(row['type']).upper() or 'DARK' in str(row['filename']).upper():
        path = os.path.join(dst, row['filename'])
        try:
            with fits.open(path) as hdul:
                dark_frames.append(hdul[0].data)
                exposure_time.append(hdul[0].header["EXPTIME"])
        except Exception as e:
            print(f"err loading {row['filename']}: {e}")

            
if dark_frames:
    for idx, i in enumerate(dark_frames):
        separated_darks.append((i - master_bias)/exposure_time[idx])
    master_dark = np.median(separated_darks, axis=0)
    print(f"master dark done using {len(dark_frames)} frames.")
    
    # plot with 2-98 percentile scaling to see structures
    plt.figure(figsize=(8, 6))
    plt.imshow(master_dark, cmap='gray', origin='lower', 
               vmin=np.percentile(master_dark, 2), vmax=np.percentile(master_dark, 98))
    plt.colorbar(label='ADU')
    plt.title('Master dark')
    plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits

# filter flats from main df
flat_df = df[
    (df['filename'].str.contains('FLAT', case=False)) | 
    (df['type'].str.contains('Flat', case=False))
].copy()

# fix missing filters by checking filename strings
for idx, row in flat_df.iterrows():
    if row['filter'] == 'N/A' or pd.isna(row['filter']):
        fname = row['filename'].upper()
        if 'RED' in fname or '_R_' in fname: flat_df.at[idx, 'filter'] = 'R'
        elif 'GREEN' in fname or '_G_' in fname: flat_df.at[idx, 'filter'] = 'G'
        elif 'BLUE' in fname or '_B_' in fname: flat_df.at[idx, 'filter'] = 'B'
        elif 'HA' in fname or 'H_ALPHA' in fname: flat_df.at[idx, 'filter'] = 'H_alpha'
        elif 'LUM' in fname: flat_df.at[idx, 'filter'] = 'Lum'

filters = flat_df['filter'].unique()
print(f"found filters: {filters}")

master_flats = {}

for filt in filters:
    if filt == 'N/A':
        continue
    print(f"\nprocessing filter: {filt}")
    frames = []
    
    filter_df = flat_df[flat_df['filter'] == filt]
    
    for idx, row in filter_df.iterrows():
        path = os.path.join(dst, row['filename'])
        try:
            with fits.open(path) as hdul:
                img_data = hdul[0].data
                exptime = hdul[0].header["EXPTIME"]
                med = np.median(img_data)
                maximum = np.max(img_data)
                
                # quick visual check
                plt.figure(figsize=(4, 3))
                plt.imshow(img_data, cmap='gray', origin='lower', 
                           vmin=np.percentile(img_data, 5), vmax=np.percentile(img_data, 95))
                plt.title(f"{row['filename']}\nMed: {med:.1f} | Max: {maximum}")
                plt.show()
                
                # skip saturated frames (ccd non-linear zone)
                if maximum > 62000:
                    print(f"skipping {row['filename']} -> saturated (max={maximum})")
                    continue
                # skip empty/underexposed frames
                if med < 800:
                    print(f"skipping {row['filename']} -> empty frame (median={med})")
                    continue
                    
                # subtract bias and dark and normalize to 1.0
                bias_sub = img_data - master_bias - (master_dark * exptime)
                norm = bias_sub / np.median(bias_sub)
                frames.append(norm)
                
        except Exception as e:
            print(f"err on {row['filename']}: {e}")
            
    if frames:
        # median combine to get rid of dust/artifacts
        master_flats[filt] = np.median(frames, axis=0)
        print(f"master flat done for filter: {filt}")
        
        # plot normalized flat (should be around 1.0)
        plt.figure(figsize=(6, 5))
        plt.imshow(master_flats[filt], cmap='gray', origin='lower', vmin=0.8, vmax=1.2)
        plt.colorbar()
        plt.title(f'Master Flat - Filter {filt}')
        plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

print("starting science frame reduction...")

# filter for actual science targets (kick out calibration files)
science_df = df[
    (~df['filename'].str.contains('BIAS|DARK|FLAT|Select|Mouse', case=False)) & 
    (~df['type'].str.contains('Bias|Flat|Dark', case=False))
]

for idx, row in science_df.iterrows():
    f_name = row['filename']
    obj = row['object']
    filt = row['filter']
    
    path = os.path.join(dst, f_name)
    
    # match master flat with current filter
    flat_frame = None
    for k in master_flats.keys():
        if str(k).upper() in str(filt).upper() or str(filt).upper() in str(k).upper():
            flat_frame = master_flats[k]
            break
            
    if flat_frame is None:
        print(f"skipping {f_name}: no master flat found for filter '{filt}'")
        if filt == "R": #technicality to make sure the R-filter frames also get aligned down the line
            out_name = "reduced_" + f_name
            out_path = os.path.join(dst, out_name)
            science_df.loc[idx, "filename"] = out_name
            with fits.open(path) as hdul:
                hdu = hdul[0]
                hdu.writeto(out_path, overwrite=True)
        continue
        
    try:
        with fits.open(path) as hdul:
            raw_data = hdul[0].data
            hdr = hdul[0].header
            exptime = hdr["EXPTIME"]
            
            # standard reduction formula: (raw - bias) / flat
            reduced = (raw_data - master_bias - (master_dark*exptime)) / flat_frame
            
            out_name = "reduced_" + f_name
            out_path = os.path.join(dst, out_name)
           
            # save new fits with original header
            hdu = fits.PrimaryHDU(reduced, header=hdr)
            hdu.writeto(out_path, overwrite=True)
            science_df.loc[idx, "filename"] = out_name
            
            print(f"reduced: {out_name} ({obj} | filter: {filt})")
            
            # plot with 2-98 scaling to check image quality
            plt.figure(figsize=(10, 8))
            vmin = np.percentile(reduced, 2)
            vmax = np.percentile(reduced, 98)
            
            plt.imshow(reduced, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
            plt.colorbar(label='intensity')
            plt.title(f"Reduced: {obj} | Filter: {filt} ({f_name})")
            plt.show()
            
    except Exception as e:
        print(f"err on reducing {f_name}: {e}")

print("all frames processed. done.")
display(science_df)

In [ ]:
#aligning
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astroalign as aa

#making some dicts to store inbetween images
align_filters = {"V":[],"B":[],"R":[]}
aligned_images = {"V":[],"B":[],"R":[]}
final = {}

for idx, row in science_df.iterrows():
    f_name = row['filename']
    obj = row['object']
    filt = row['filter']
    
    path = os.path.join(dst, f_name)
    
    if "reduced" in f_name:
        with fits.open(path) as hdul:
            align_filters[filt].append(hdul[0].data)

images = align_filters["V"]
print(np.shape(images))
target = np.asarray(images)[6,:,:]
vmin = np.percentile(target, 2)
vmax = np.percentile(target, 98)
plt.imshow(target, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
plt.colorbar(label='intensity')
plt.show()

In [ ]:
#aligning
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astroalign as aa

#making some dicts to store inbetween images
align_filters = {"V":[],"B":[],"R":[]}
aligned_images = {"V":[],"B":[],"R":[]}
final = {}


for idx, row in science_df.iterrows():
    f_name = row['filename']
    obj = row['object']
    filt = row['filter']
    
    path = os.path.join(dst, f_name)
    
    if "reduced" in f_name:
        with fits.open(path) as hdul:
            align_filters[filt].append(hdul[0].data)
    
#aligning the stacks in each filter to an arbitrary image
for key, images in align_filters.items():
    print(np.shape(images))
    if key == "R":
        target = np.asarray(images)[4,:,:]
    else:
        target = np.asarray(images)[6,:,:]
    print(np.shape(target))
    plt.imshow(target)
    for index, image in enumerate(images):
        print(np.shape(image))
        al_image = np.ascontiguousarray(image, dtype=np.float64)
        al_target = np.ascontiguousarray(target, dtype=np.float64)
        try:
            aligned_im, _ = aa.register(al_image, al_target, detection_sigma = 3.0, max_control_points = 100, min_area = 5)
            aligned_images[key].append(aligned_im)
        except:
            print(f"Could not align {key} frame: {index}")

plt.figure(figsize=(10, 8))
vmin = np.percentile(reduced, 2)
vmax = np.percentile(reduced, 98)

#stack the images into 1 image per filter and display them
for key, images in aligned_images.items():
    print(np.shape(images))
    final[key] = np.median(images, axis=0)
    print(np.shape(final[key]))
    hdu = fits.PrimaryHDU(final[key], header=hdr)
    hdu.writeto("Final_{key}.fits", overwrite=True)
    plt.imshow(final[key], cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
    plt.colorbar(label='intensity')
    plt.title(f"Stacked Filter: {key}")
    plt.show()
            

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, aperture_photometry
from scipy.spatial import KDTree


# Adjust this directory path to where your final FITS files are stored
DATA_DIR = '/net/virgo01/data/users/ahayr/meine_beobachtungen/'

def load_stacked_image(filter_key):
    """
    Attempts to load a stacked image. First looks into the active RAM (final dict),
    and if not found, falls back to searching for FITS files on the disk.
    
    Parameters:
        filter_key (str): The astronomical filter name, e.g., 'B' or 'V'.
        
    Returns:
        np.ndarray: The 2D image data as floats.
    """
    # Attempt 1: Try retrieving the image from RAM (variables from previous notebook cells)
    try:
        img = final[filter_key]
        print(f"[{filter_key}] Successfully loaded from RAM. Shape: {img.shape}")
        return img
    except NameError:
        # 'final' dictionary does not exist in local scope, moving to fallback
        pass

    # Attempt 2: Search for the FITS file on disk using multiple naming conventions
    file_candidates = [
        os.path.join(DATA_DIR, f'Final_{filter_key}.fits'),
        os.path.join(DATA_DIR, f'Final_{filter_key}.fit'),
        os.path.join(DATA_DIR, f'final_{filter_key}.fits'),
    ]
    
    for path in file_candidates:
        if os.path.exists(path):
            with fits.open(path) as hdul:
                # Convert data to float to prevent underflow/overflow during background subtraction
                img = hdul[0].data.astype(float)
            print(f"[{filter_key}] Loaded from disk: {path}. Shape: {img.shape}")
            return img

    # Exception Handling: Triggered if the image is found neither in RAM nor on Disk
    raise FileNotFoundError(
        f"No stacked image found for filter '{filter_key}'.\n"
        f"Checked paths: {file_candidates}\n"
        f"Please run 'Cell 9' (Image Stacking) before executing this cell."
    )

# Execute the loading function for both the Blue (B) and Visual (V) filters
stacked_B = load_stacked_image('B')
stacked_V = load_stacked_image('V')


def extract_magnitudes(image, fwhm=4.0, threshold_sigma=5.0, label=''):
    """
    Detects stars using the DAOStarFinder algorithm and performs aperture photometry 
    to calculate instrumental magnitudes.
    
    Parameters:
        image (np.ndarray): 2D array of the astronomical image.
        fwhm (float): Full Width at Half Maximum of the stellar profiles in pixels.
        threshold_sigma (float): S/N threshold factor above the background noise.
        label (str): Identifier used for logging/printing (e.g., 'B' or 'V').
        
    Returns:
        astropy.table.Table: Catalog containing detected sources and their magnitudes.
    """
    # Step 2.1: Background estimation using Sigma Clipping (rejects bright outliers like stars)
    mean, median, std = sigma_clipped_stats(image, sigma=3.0)
    
    # Subtract the median background level to create a flat baseline for star detection
    img_subtraction = image - median

    # Step 2.2: Initialize the DAOStarFinder algorithm 
    # The threshold for detection is defined as: factor * standard_deviation_of_background
    daofind = DAOStarFinder(fwhm=fwhm, threshold=threshold_sigma * std)
    sources = daofind(img_subtraction)

    # Error handling if the algorithm yields zero results
    if sources is None or len(sources) == 0:
        raise RuntimeError(
            f"[{label}] Error: No stars detected!\n"
            f"  Background Median={median:.1f}, Std={std:.1f}, Detection Threshold={threshold_sigma*std:.1f}\n"
            f"  Recommendation: Try adjusting the 'fwhm' or lowering 'threshold_sigma'."
        )
    print(f"[{label}] Detection successful: Found {len(sources)} stars.")

    # Step 2.3: Define circular apertures around the detected star centers
    # Usually, a radius of 1.5 to 2 times the FWHM captures most of the stellar flux
    positions = np.transpose((sources['x_centroid'], sources['y_centroid']))
    apertures = CircularAperture(positions, r=fwhm * 1.5)
    
    # Perform the photometry (summing up pixel values inside each circle)
    photometry_table = aperture_photometry(img_subtraction, apertures)

    # Step 2.4: Clean the dataset
    # Filter out negative flux values caused by edge artifacts, bad pixels, or noise fluctuations
    photometry_table = photometry_table[photometry_table['aperture_sum'] > 0]

    # Step 2.5: Calculate Instrumental Magnitude
    # Formula: mag = -2.5 * log10(flux) + ZeroPoint
    # We use an arbitrary zero point of 25.0 to keep magnitudes positive and relative
    photometry_table['mag'] = -2.5 * np.log10(photometry_table['aperture_sum']) + 25.0

    # Step 2.6: Standardize column names to guarantee compatibility with cross-matching steps later
    if 'xcenter' in photometry_table.colnames:
        photometry_table.rename_column('xcenter', 'x_center')
        photometry_table.rename_column('ycenter', 'y_center')

    return photometry_table



# Run photometry on both images. 
# Note: You can tweak 'fwhm' (typically 4-6 px) and 'threshold_sigma' (typically 3-7) if needed.
catalog_B = extract_magnitudes(stacked_B, fwhm=5.0, threshold_sigma=5.0, label='B')
catalog_V = extract_magnitudes(stacked_V, fwhm=5.0, threshold_sigma=5.0, label='V')

# Print summary results
print(f"\nFinal Summary:")
print(f"--> Catalog B contains {len(catalog_B)} confirmed stars.")
print(f"--> Catalog V contains {len(catalog_V)} confirmed stars.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree


# Maximum allowed distance in pixels to consider two detections as the same star
MATCH_TOLERANCE_PIX = 3  

# Extract (X, Y) coordinates from both catalogs and stack them into 2D arrays
coords_B = np.array(list(zip(
    catalog_B['x_center'].value,
    catalog_B['y_center'].value
)))
coords_V = np.array(list(zip(
    catalog_V['x_center'].value,
    catalog_V['y_center'].value
)))

# Build a K-d Tree using the V-filter coordinates for fast spatial nearest-neighbor lookups
tree_V = KDTree(coords_V)

# Initialize empty lists to store the magnitudes of matching pairs
matched_B_mags, matched_V_mags = [], []

# Loop through each star in the B-catalog and find its nearest neighbor in the V-catalog
for i, coord in enumerate(coords_B):
    # query() returns the distance to the nearest neighbor and its index in coords_V
    distance, index = tree_V.query(coord)
    
    # If the neighbor is within our pixel tolerance, we accept it as a valid match
    if distance < MATCH_TOLERANCE_PIX:
        matched_B_mags.append(float(catalog_B['mag'][i]))
        matched_V_mags.append(float(catalog_V['mag'][index]))

# Convert the results into numpy arrays for vector math operations
matched_B = np.array(matched_B_mags)
matched_V = np.array(matched_V_mags)

# Calculate the instrumental B-V color index (the x-axis of our CMD)
color_index_BV = matched_B - matched_V

# Print matching summary and warnings if the dataset is critically small
print(f"Cross-matching complete. Identical stars found: {len(matched_V)}")
if len(matched_V) < 10:
    print("Warning: Very few matches found! Consider increasing 'MATCH_TOLERANCE_PIX' or verifying the FWHM in your detection step.")


# Set up the figure size (a vertical layout is preferred for Hertzsprung-Russell / CM Diagrams)
fig, ax = plt.subplots(figsize=(7, 9))

# Create the scatter plot: X-axis = Color (B-V), Y-axis = Magnitude (V)
ax.scatter(color_index_BV, matched_V, s=8, alpha=0.5, color='steelblue', label='Stars')

# CRITICAL STEP: Astronomers invert the Y-axis because lower magnitudes mean brighter stars!
ax.invert_yaxis()

# Labeling and styling the plot
ax.set_xlabel('B – V (instrumental)', fontsize=13)
ax.set_ylabel('V (instrumental)', fontsize=13)
ax.set_title('Color-Magnitude Diagram – M52 (Instrumental)', fontsize=14)
ax.grid(True, linestyle=':', alpha=0.4)
ax.legend()

# Optimize layout to prevent text clipping, then save and display
plt.tight_layout()
plt.savefig('CMD_instrumental.png', dpi=150)
plt.show()

print("Plot successfully saved to disk as 'CMD_instrumental.png'")

In [ ]:
import gzip
import io
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ISOCHRONE_FILE = 'isochrone_Z0152.dat.gz'
ISOCHRONE_DIR = '/net/virgo01/data/users/ahayr/Observational Astronomy/'

# Logarithmic target ages to extract and plot later (e.g., 8.0 = 10^8 years = 100 Myr)
LOG_AGES_TO_PLOT = [7.7, 8.0, 8.3, 8.6]


isochrone_path = os.path.join(ISOCHRONE_DIR, ISOCHRONE_FILE)
assert os.path.exists(isochrone_path), f"File not found: {isochrone_path}"

# Dynamically choose the correct file opener based on the file extension
open_function = gzip.open if isochrone_path.endswith('.gz') else open

# Read all lines into memory to isolate metadata and headers
with open_function(isochrone_path, 'rt') as file:
    all_lines = file.readlines()

# Extract the header columns: In standard stellar evolutionary tracks (like PARSEC/MIST),
# the last commented line containing '#' right before the actual data data rows holds the column keys.
header_line = ''
for line in all_lines:
    if line.startswith('#'):
        header_line = line
    else:
        # Stop scanning once we reach the first actual numerical data row
        break

# Clean up comment symbols and split line into individual column strings
column_names = header_line.lstrip('#').split()
print("Detected Isochrone Columns:", column_names)

# Convert the raw lines back to a virtual file stream via StringIO and load it with pandas
# 'sep=r"\s+"' handles variable whitespace spacing between the text data columns
isochrone_df = pd.read_csv(
    io.StringIO(''.join(all_lines)), 
    comment='#', 
    sep=r'\s+', 
    names=column_names
)
print(f"Successfully loaded isochrone. Total rows: {len(isochrone_df)}")
print("Preview of the first 3 rows:")
print(isochrone_df.head(3))



def find_column_by_candidates(df, candidate_names):
    """
    Scans a dataframe to locate a column matching a list of common astronomical naming conventions.
    Maintains flexibility across different synthetic stellar catalogs (e.g., PARSEC vs. MIST).
    """
    for candidate in candidate_names:
        if candidate in df.columns:
            return candidate
    raise KeyError(
        f"None of the requested candidate columns {candidate_names} were found.\n"
        f"Available keys in this file: {list(df.columns)}"
    )

# Map filters dynamically using common syntax structures found in file templates
COL_V = find_column_by_candidates(isochrone_df, ['Vmag', 'V', 'V_mag', 'mV'])
COL_B = find_column_by_candidates(isochrone_df, ['Bmag', 'B', 'B_mag', 'mB'])
COL_AGE = find_column_by_candidates(isochrone_df, ['logAge', 'log(age/yr)', 'logA', 'lage'])

print(f"\nMapping successfully assigned: V_Filter='{COL_V}', B_Filter='{COL_B}', Log_Age='{COL_AGE}'")


# Calculate the absolute B-V color index for the stellar models (X-axis for our theoretical tracks)
isochrone_df['B_V_abs'] = isochrone_df[COL_B] - isochrone_df[COL_V]

# Round values to filter unique logAge steps included inside this model grid
unique_log_ages = np.unique(np.round(isochrone_df[COL_AGE], 2))
print(f"Available log(Age) values within this model grid:\n{unique_log_ages}")

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------------------------------------------------------
# 1. SLIDER PARAMETERS – Adjust these values to shift the theoretical tracks
#    until the isochrone aligns perfectly with the observed Main Sequence (MS).
# -----------------------------------------------------------------------------

# Vertical shift: Accounts for distance modulus + visual extinction (m_V - M_V)
DELTA_V = 13.3   

# Horizontal shift: Accounts for interstellar reddening, E(B-V) = (B-V)_obs - (B-V)_intrinsic
DELTA_BV = 0.72  

# Standard interstellar extinction law ratio (R_V = A_V / E(B-V))
# The standard value for the diffuse interstellar medium in the Milky Way is 3.1
R_V = 3.1

# -----------------------------------------------------------------------------
# 2. Derived Physical Properties
# -----------------------------------------------------------------------------

# Interstellar Reddening (Color Excess)
color_excess_E_BV = DELTA_BV

# Total Visual Extinction (dimming of light in the V-band caused by interstellar dust)
extinction_A_V = R_V * color_excess_E_BV

# True Distance Modulus: mu = m - M = 5 * log10(d / 10pc)
# We subtract the dust extinction (A_V) from our total vertical shift to get the true distance
distance_modulus = DELTA_V - extinction_A_V

# Convert distance modulus to actual physical distance in parsecs (pc) using: d = 10^((mu + 5) / 5)
distance_pc = 10 ** (distance_modulus / 5 + 1)

# Print a formatted physical summary block to the console
print("=" * 50)
print(f"  Color Excess E(B–V)   = {color_excess_E_BV:.3f} mag  (Reddening)")
print(f"  Visual Extinction A_V = {extinction_A_V:.3f} mag  (Dust Dimming)")
print(f"  True Distance Modulus = {distance_modulus:.2f} mag")
print(f"  Calculated Distance   = {distance_pc:.0f} pc  ({distance_pc/1000:.2f} kpc)")
print("=" * 50)


# -----------------------------------------------------------------------------
# 3. Visualization: CMD + Shifted Isochrone(n)
# -----------------------------------------------------------------------------

# Generate a sequential colormap (Plasma) to visually distinguish different stellar ages
isochrone_colors = cm.plasma(np.linspace(0.1, 0.9, len(LOG_AGES_TO_PLOT)))

fig, ax = plt.subplots(figsize=(8, 10))

# Plot the real observed star cluster data
ax.scatter(
    color_index_BV, 
    matched_V, 
    s=8, 
    alpha=0.45, 
    color='steelblue',
    zorder=2, 
    label='Observed Stars'
)

# Standardize axis boundaries to focus on the main sequence and turn-off point
ax.set_xlim(-1, 4)   # Typical B-V range for young to intermediate open clusters
ax.set_ylim(22, 8)   # V magnitude scale: from 22 (faint) down to 8 (bright)

# Overlay the theoretical isochrone tracks shifted by our parameters
for log_age, color in zip(LOG_AGES_TO_PLOT, isochrone_colors):
    
    # Locate the closest available age grid step in the loaded isochrone dataframe
    closest_age = unique_log_ages[np.argmin(np.abs(unique_log_ages - log_age))]
    subset = isochrone_df[np.abs(isochrone_df[COL_AGE] - closest_age) < 0.01].copy()

    # Filter out post-main-sequence evolutionary phases if the catalog uses phase labels 
    # (Label <= 4 usually isolates Main Sequence and Subgiant branches to reduce plot clutter)
    if 'label' in subset.columns:
        subset = subset[subset['label'] <= 4]

    if subset.empty:
        print(f"Warning: No valid data found for isochrone logAge={log_age}")
        continue

    # Apply the shifts: Shift absolute magnitudes to apparent instrumental magnitudes
    V_shifted = subset[COL_V] + DELTA_V
    BV_shifted = subset['B_V_abs'] + DELTA_BV

    # Convert log(Age) back to standard years and then to Megayears (Myr) for the plot legend
    age_myr = (10 ** closest_age) / 1e6
    ax.plot(
        BV_shifted, 
        V_shifted, 
        lw=1.8, 
        color=color,
        label=f'Isochrone {age_myr:.0f} Myr (logAge={closest_age:.1f})', 
        zorder=3
    )

# Standard astronomical convention: smaller/fainter magnitudes belong at the bottom
ax.invert_yaxis()

# Labeling and metadata layout
ax.set_xlabel('B – V (apparent)', fontsize=13)
ax.set_ylabel('V (instrumental)', fontsize=13)
ax.set_title(
    f'CMD M52 with Isochrone Fits\n'
    f'$\mu$={distance_modulus:.1f} mag | d={distance_pc:.0f} pc | '
    f'E(B–V)={color_excess_E_BV:.2f} | A_V={extinction_A_V:.2f}',
    fontsize=12
)

ax.legend(fontsize=9, loc='upper right')
ax.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()

# Save the calibrated plot to disk
plt.savefig('CMD_with_isochrone.png', dpi=150)
plt.show()

print("Plot successfully saved to disk as 'CMD_with_isochrone.png'")

In [ ]:
# Reference values taken from standard astronomical catalogs/literature for validation
literature_distance_pc = 1400      # Approximate distance (uncertainty roughly ±100 pc)
literature_E_BV = 0.63             # Standard interstellar reddening value
literature_age_myr = 100           # Estimated cluster age (~100 Megayears, logAge ≈ 8.0)

# Print a structured ASCII comparison table to contrast experimental results with literature
print("╔══════════════════════════════════════════════════════╗")
print("║             FINAL CLUSTER PARAMETERS – M52           ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Distance         {distance_pc:>7.0f} pc   (Lit: ~{literature_distance_pc} pc)       ║")
print(f"║  Distance Modulus {distance_modulus:>7.2f} mag                           ║")
print(f"║  E(B–V) Reddening {color_excess_E_BV:>7.3f} mag   (Lit: ~{literature_E_BV:.2f})          ║")
print(f"║  A_V Extinction   {extinction_A_V:>7.3f} mag                           ║")
print(f"║  Cluster Age      200–400 Myr   (Lit: ~{literature_age_myr} Myr)         ║")
print("║  Metallicity      ~ Solar (Z=0.0152)                 ║")
print("╚══════════════════════════════════════════════════════╝")
print()


In [ ]:
import subprocess
result = subprocess.run(
    ['jupyter', 'nbconvert', '--clear-output', '--inplace', '__file__'],
    capture_output=True, text=True
)
# Pfad anpassen:
import os
path = '/net/virgo01/data/users/ahayr/Observational Astronomy/HR1.ipynb/'  # <── anpassen
result = subprocess.run(
    ['jupyter', 'nbconvert', '--clear-output', '--inplace', path],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import os
path = '/net/virgo01/data/users/ahayr/Observational Astronomy/'
print([f for f in os.listdir(path) if f.endswith('.ipynb')])